<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-10-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-10-10: Statistics — Confounded Drift Across Segments

**Dataset:** [Palmer Penguins](https://github.com/mwaskom/seaborn-data) — 344 real penguins measured at
three research stations in the Palmer Archipelago, Antarctica (bill length, bill depth, flipper length,
body mass, sex, species, island), fetched live from seaborn's public dataset mirror.

## Problem statement

A single regression model that predicts `body_mass_g` from bill and flipper measurements is about to be
deployed across three field stations: **Biscoe**, **Dream**, and **Torgersen** (the `island` column). Before
shipping one global model, QA needs an answer: do the four numeric input features show a statistically
significant distribution difference between islands? If they do, is a single global model still safe, or
does each island need its own model?

There's a trap built into this dataset, and it mirrors a real production failure mode: the three islands
don't host the same mix of species. **Gentoo** penguins only live on Biscoe, **Chinstrap** only on Dream,
while **Adelie** penguins are spread across all three. So any statistical difference you find between
islands could be a genuine per-island effect — or it could just be species composition leaking through,
the same way a "regional drift" alarm in production is sometimes really just a change in *who* uses the
product, not a change in how any one segment behaves.

## What you should produce

1. `pairwise_feature_tests(frame, group_col, feature_cols, alpha=0.05)` — for every (feature, pair of
   groups) combination, run a two-sided Mann–Whitney U test and return a tidy `DataFrame` with columns
   `feature`, `group_a`, `group_b`, `u_stat`, `p_value`, `p_value_bonferroni`, `flagged_raw`,
   `flagged_corrected`.
2. Run that function on the **full cleaned dataset**, grouped by `island`, across all four numeric
   features. Report how many of the 12 tests are flagged, raw vs. Bonferroni-corrected.
3. Run it again on an **Adelie-only subset** (the one species present on every island), grouped by
   `island`. Report the same counts.
4. `bootstrap_ci_mean_diff(a, b, n_boot=5000, ci=0.95, seed=0)` — a percentile bootstrap confidence
   interval for the difference in means between two samples. Apply it to `body_mass_g` for
   Biscoe vs. Dream, once on the full dataset and once on the Adelie-only subset, so you can see the
   effect size itself shrink, not just the p-value.
5. A short written conclusion (3–5 sentences) answering: is it safe to ship one global model across all
   three islands? What did Bonferroni correction actually change here, and why would a drift check that
   skipped step 3 (controlling for species) have produced a misleading conclusion?


## Setup

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats

RANDOM_STATE = 0
pd.set_option("display.width", 120)

## Load and clean the data

Loaded straight from the public CSV mirror. A handful of rows are missing measurements or `sex` —
dropped here as a basic data-validation step, since a null in a feature the model needs is not something
a hypothesis test downstream should have to deal with.

In [ ]:
PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

NUMERIC_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

raw = pd.read_csv(PENGUINS_URL)
penguins = raw.dropna(subset=NUMERIC_FEATURES + ["sex"]).reset_index(drop=True)

print(penguins.shape)
print(pd.crosstab(penguins["species"], penguins["island"]))
penguins.head()

## Your task

Fill in the TODOs below. Keep the function signatures as given — the write-up cells assume these names.

In [ ]:
def pairwise_feature_tests(frame, group_col, feature_cols, alpha=0.05):
    """TODO: for every feature in feature_cols, run a two-sided Mann-Whitney U test
    (scipy.stats.mannwhitneyu) between every pair of groups in frame[group_col].

    Steps:
    1. Get the sorted unique values of frame[group_col] and all pairwise combinations
       (itertools.combinations with r=2).
    2. For each (feature, group_a, group_b) combination, run mannwhitneyu(a, b,
       alternative="two-sided") where a/b are the feature values for each group.
    3. Collect rows with keys: feature, group_a, group_b, u_stat, p_value.
    4. Build a DataFrame from the rows. Let m = number of rows (number of tests run).
       Add a p_value_bonferroni column = min(p_value * m, 1.0) for every row.
    5. Add flagged_raw = p_value < alpha, flagged_corrected = p_value_bonferroni < alpha.
    6. Return the DataFrame sorted by p_value ascending.
    """
    raise NotImplementedError

In [ ]:
# TODO: call pairwise_feature_tests on the full `penguins` DataFrame, grouped by "island",
# across NUMERIC_FEATURES. Store it as `full_report` and print how many rows have
# flagged_raw == True vs. flagged_corrected == True (out of how many total tests).

full_report = None

In [ ]:
# TODO: build `adelie`, the subset of `penguins` where species == "Adelie", then call
# pairwise_feature_tests on it the same way as above. Store it as `adelie_report` and
# print the same flagged_raw / flagged_corrected counts.

adelie = None
adelie_report = None

In [ ]:
def bootstrap_ci_mean_diff(a, b, n_boot=5000, ci=0.95, seed=0):
    """TODO: percentile bootstrap confidence interval for mean(b) - mean(a).

    Steps:
    1. Convert a, b to numpy arrays. Create a numpy random Generator from `seed`.
    2. Repeat n_boot times: resample a and b independently, with replacement, at their
       original sizes, and record the difference of resample means (b_resample.mean() -
       a_resample.mean()).
    3. Return a tuple (point_estimate, ci_low, ci_high) where point_estimate is the mean
       of the n_boot differences, and (ci_low, ci_high) are the percentiles of the
       bootstrap distribution corresponding to the requested `ci` (e.g. 2.5th/97.5th for
       ci=0.95).
    """
    raise NotImplementedError

In [ ]:
# TODO: compute bootstrap_ci_mean_diff for body_mass_g, Biscoe vs. Dream, twice:
#   1. biscoe_dream_full: using the full `penguins` dataset
#   2. biscoe_dream_adelie: using the `adelie`-only subset
# Print both so you can compare the effect size, not just whether it's "significant".

biscoe_dream_full = None
biscoe_dream_adelie = None

### Write-up

TODO: in 3–5 sentences, answer: is it safe to ship one global `body_mass_g` model across all three
islands? What did the Bonferroni correction actually change between the full-dataset and Adelie-only
results? And why would a drift check that only ran step 2 (full dataset, no species control) have been
misleading on its own?

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def pairwise_feature_tests(frame, group_col, feature_cols, alpha=0.05):
    groups = sorted(frame[group_col].unique())
    rows = []
    for feat in feature_cols:
        for g1, g2 in combinations(groups, 2):
            a = frame.loc[frame[group_col] == g1, feat]
            b = frame.loc[frame[group_col] == g2, feat]
            u_stat, p_value = stats.mannwhitneyu(a, b, alternative="two-sided")
            rows.append({
                "feature": feat, "group_a": g1, "group_b": g2,
                "u_stat": u_stat, "p_value": p_value,
            })
    result = pd.DataFrame(rows)
    m = len(result)
    result["p_value_bonferroni"] = (result["p_value"] * m).clip(upper=1.0)
    result["flagged_raw"] = result["p_value"] < alpha
    result["flagged_corrected"] = result["p_value_bonferroni"] < alpha
    return result.sort_values("p_value").reset_index(drop=True)


full_report = pairwise_feature_tests(penguins, "island", NUMERIC_FEATURES)
print("Full dataset:", full_report["flagged_raw"].sum(), "/", len(full_report), "raw flags,",
      full_report["flagged_corrected"].sum(), "/", len(full_report), "corrected flags")

adelie = penguins[penguins["species"] == "Adelie"].reset_index(drop=True)
adelie_report = pairwise_feature_tests(adelie, "island", NUMERIC_FEATURES)
print("Adelie-only:", adelie_report["flagged_raw"].sum(), "/", len(adelie_report), "raw flags,",
      adelie_report["flagged_corrected"].sum(), "/", len(adelie_report), "corrected flags")


def bootstrap_ci_mean_diff(a, b, n_boot=5000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs[i] = sb.mean() - sa.mean()
    tail = (1 - ci) / 2 * 100
    lo, hi = np.percentile(diffs, [tail, 100 - tail])
    return diffs.mean(), lo, hi


biscoe_full = penguins.loc[penguins["island"] == "Biscoe", "body_mass_g"]
dream_full = penguins.loc[penguins["island"] == "Dream", "body_mass_g"]
biscoe_dream_full = bootstrap_ci_mean_diff(biscoe_full, dream_full)

biscoe_adelie = adelie.loc[adelie["island"] == "Biscoe", "body_mass_g"]
dream_adelie = adelie.loc[adelie["island"] == "Dream", "body_mass_g"]
biscoe_dream_adelie = bootstrap_ci_mean_diff(biscoe_adelie, dream_adelie)

print("Biscoe vs Dream, full dataset:   mean diff =", biscoe_dream_full)
print("Biscoe vs Dream, Adelie only:    mean diff =", biscoe_dream_adelie)
```

**What happens when you run this:**

On the full dataset, roughly 8 of the 12 Mann–Whitney tests come back significant, raw *and* after
Bonferroni correction (`m=12` is small enough that the correction barely moves anything — it divides
every p-value by 12, and the real effects here have p-values many orders of magnitude below 0.05/12).
That alone would look like strong evidence of per-island drift in all four features.

On the Adelie-only subset — the one species measured on all three islands — **every single test comes
back non-significant**. The island "effect" on the full dataset wasn't really an island effect at all:
Biscoe is ~75% Gentoo (the largest-bodied species), Dream is ~55% Chinstrap, and once you hold species
fixed, flipper length, bill shape, and body mass barely move between stations. The bootstrap confidence
interval tells the same story in effect-size terms: the Biscoe-vs-Dream body-mass gap on the full dataset
is roughly −1000g with a tight CI nowhere near zero, while the same comparison restricted to Adelie
shrinks to a CI that comfortably straddles zero.

**Takeaway:** a single global model is very plausibly fine here — the earlier "drift" signal was a
confound (species mix), not a genuine per-island distribution shift in the features themselves. Bonferroni
correction mattered less than the subgroup control did. In production, the equivalent mistake is running
a feature-drift check across "US vs. EU traffic" without first checking whether the two regions also
differ in device mix, account age, or plan tier — you can spend a long time chasing a "regional drift"
alert that is actually just a population-composition change, unless you slice by the confounding variable
before you trust the test.

</details>